# Teil 4: Evaluation des Modells

In [16]:
import pandas as pd

data = pd.read_csv("student_dropout_cleaned.csv")

data.head()

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0
4,7,24.5,Male,25000.0,Yes,3.00,78.2,1,37.4,Yes,Yes,7.3,0.64,0.33,0.44,Year 4,CS,Bachelor,0


## 4.1 Wichtige Felder für das Modell

Um herauszufinden, welche Felder für das Modell besonders aussagekräftig sind, wird ein DecisionTreeClassifier erneut trainiert. Danach werden die Feature Importances ausgegeben. Diese zeigen, welche Eingabefelder für die Entscheidungen des Modells am wichtigsten waren.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

y = data["Dropout"]

X = data.drop(columns=["Dropout", "Student_ID"])
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    "Feld": X.columns,
    "Wichtigkeit": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Wichtigkeit", ascending=False
)

feature_importance.head(10)

,Feld,Wichtigkeit
7,GPA,0.296197
5,Travel_Time_Minutes,0.095895
3,Attendance_Rate,0.095007
2,Study_Hours_per_Day,0.084873
6,Stress_Index,0.068636
0,Age,0.067628
8,Semester_GPA,0.052389
1,Family_Income,0.044104
9,CGPA,0.042910
4,Assignment_Delay_Days,0.032943


Die Tabelle zeigt die wichtigsten Felder für das Modell. Je höher der Wert bei Wichtigkeit ist, desto stärker hat dieses Feld die Entscheidungen des DecisionTreeClassifier beeinflusst. Besonders die obersten Felder sind für die Vorhersage des Studienabbruchs wichtig. Diese Werte werden später genutzt, um besser zu verstehen, warum das Modell bestimmte Vorhersagen trifft.

## 4.2 Messmetrik

Da es sich bei Dropout um ein Klassifikationsproblem handelt, wird die Accuracy als Messmetrik verwendet. Die Accuracy zeigt, wie viele Vorhersagen des Modells insgesamt richtig waren.

In [18]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.7161862527716186


Die Accuracy zeigt den Anteil der richtig vorhergesagten Fälle. Ein Wert von 0.70 würde zum Beispiel bedeuten, dass 70 Prozent der Testdaten korrekt vorhergesagt wurden. Die Accuracy gibt einen ersten Überblick über die Modellqualität, reicht alleine aber nicht aus. Deshalb wird zusätzlich eine Wahrheitsmatrix berechnet.

## 4.3 Wahrheitsmatrix, Sensitivität und Spezifizität

Mit einer Wahrheitsmatrix kann genauer untersucht werden, welche Fälle richtig und falsch vorhergesagt wurden. Daraus werden zusätzlich Sensitivität und Spezifizität berechnet.

In [19]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Wahrheitsmatrix:")
print(cm)
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)
print("Sensitivität:", sensitivity)
print("Spezifizität:", specificity)

Wahrheitsmatrix:
[[1111  258]
 [ 254  181]]
True Negatives: 1111
False Positives: 258
False Negatives: 254
True Positives: 181
Sensitivität: 0.4160919540229885
Spezifizität: 0.8115412710007305


Die Wahrheitsmatrix zeigt, dass das Modell Nicht-Dropouts deutlich besser erkennt als tatsächliche Dropouts. Die Spezifizität liegt bei ungefähr 0.81, was bedeutet, dass viele Personen ohne Dropout korrekt erkannt werden. Die Sensitivität liegt jedoch nur bei ungefähr 0.42. Das bedeutet, dass viele echte Dropout-Fälle vom Modell nicht erkannt werden. Das Modell ist deshalb für die Erkennung von Dropouts noch nicht sehr zuverlässig.

## 4.4 Fazit zur Modellqualität

Das Modell funktioniert grundsätzlich, ist aber noch nicht sehr zuverlässig. Die Accuracy gibt einen ersten positiven Eindruck, jedoch zeigt die Wahrheitsmatrix ein genaueres Bild. Besonders die Sensitivität ist eher tief, da viele tatsächliche Dropout-Fälle nicht erkannt werden. Die Spezifizität ist besser, weil Nicht-Dropouts häufiger korrekt vorhergesagt werden. Ein möglicher Grund dafür ist, dass die Klassen im Datensatz nicht gleich stark vertreten sind oder dass ein einfacher Entscheidungsbaum zu ungenau ist.